In [13]:
import google.generativeai as genai
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import os
from tqdm import tqdm
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

In [14]:
prompt = """You are an expert at extracting data from chart images and converting it into CSV.

Process:

1. Image Analysis: Examine the chart. Identify:
	* Title
	* X-Axis Labels
	* Y-Axis Labels (including units, if present or inferable)
	* Data Points (numerical values matched to graph location).

2. Reasoning: Explain your extraction, including:
	* How you identified values and headers.
	* Reasoning for any inferred labels/units.
    
3. Table Generation: Create CSV table.
	* First row: Headers (based on extracted info).
	* Remaining rows: Data points, one row per data point.

4. Chart Types:
	* Line/Bar/Scatter: Columns: Chart Title, X-Axis values, Y-Axis Values, Units.
	* Pie: Columns: "Chart Title", "Category Name", "Value".
	* Box Plot: Columns: "Chart Title", "1st Quartile", "2nd Quartile", "3rd Quartile".
	* Other: Follow the general process and use appropriate CSV format.

5. Column Headers:
	* Base headers on extracted information.
    * Include chart title in the first column as 'Chart Title'.

6. CSV Formatting:
	* Commas to separate values.
	* Double quotes for values with commas.
	* Newlines to separate rows.
    * Ensure the chart title is always present in the first column.

7. Accuracy Check: Ensure data points are correctly placed and headers are descriptive.

Output Format:

<reasoning>
[Detailed explanation of data extraction.]
</reasoning>
<table>
[CSV formatted table.]
</table>

Example:

<reasoning>
Chart has title:'USA'. Chart has 'Year' on X, 'Spending' on Y in 'Million USD'. Data extracted. Headers are 'Year', 'Spending', 'Units'.
Chart Title,Year,Spending,Units
USA,2020,3.3,Million USD
USA,2021,3.5,Million USD
USA,2022,3.7,Million USD
</reasoning>
<table>
Chart Title,Year,Spending,Units
USA,2020,3.3,Million USD
USA,2021,3.5,Million USD
USA,2022,3.7,Million USD
</table>
"""

In [ ]:
api_keys= []

In [16]:
genai.configure(api_key=api_keys[0])

generation_config = {
  "temperature": 1,
  "max_output_tokens": 8192,
}
safety_settings = [
  {
	"category": "HARM_CATEGORY_HARASSMENT",
	"threshold": "BLOCK_NONE"
  },
  {
	"category": "HARM_CATEGORY_HATE_SPEECH",
	"threshold": "BLOCK_NONE"
  },
  {
	"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT",
	"threshold": "BLOCK_NONE"
  },
  {
	"category": "HARM_CATEGORY_DANGEROUS_CONTENT",
	"threshold": "BLOCK_NONE"
  }
]

model = genai.GenerativeModel(
	# model_name="gemini-2.0-flash-exp",
	model_name="gemini-1.5-pro-002",
	safety_settings=safety_settings, 
	generation_config=generation_config
)

In [17]:
df = pd.read_json('../type1/qa.json')
indices = df['image_index'].unique().astype(str)
all_names = sorted(os.listdir('../type1/simple/'))
curr_ind = 0
charts = defaultdict(list) 
for name in all_names:
	charts[name.split('_')[1]].append(name)    

In [18]:
def get_output(image):
	try:
		answer = model.generate_content([prompt, image]).text
		return answer
	except Exception as e:
		print(e)
		return None
	
def multithread_get_output(list_of_imgs):
	with ThreadPoolExecutor(max_workers = 20) as executor:
		return list(executor.map(get_output, list_of_imgs))

In [19]:
index_to_img = {}
model_name = 'gemini-1.5-pro-002'

for i, image_name in enumerate(tqdm(indices)):
	all_related_imgs = []
	for chart_name in charts[image_name]:
		chart_num = chart_name.split('_')[-1].split('.')[0]
		image_path = '../type1/simple/' + chart_name
		image = Image.open(image_path)
		all_related_imgs.append(image)
	index_to_img[image_name] = all_related_imgs

100%|██████████| 355/355 [00:00<00:00, 3305.17it/s]


In [20]:
model.generate_content("hi").text

'Hi there! How can I help you today?\n'

In [22]:
operation_counter = 0
model_name = 'gemini_1.5_pro'
print(len(all_names))

last_found = -1

if os.path.exists(f'../model_responses/type1/baselines/{model_name}_chart_to_table/'):
	last_found = sorted([int(i) for i in os.listdir(f'../model_responses/type1/baselines/{model_name}_chart_to_table/')])[-1]

for i, image_name in enumerate(tqdm(indices)):
	if last_found != -1 and int(image_name) < int(last_found):
		continue
	outputs = multithread_get_output(index_to_img[image_name])
	for chart_num, answer in enumerate(outputs):
		table = answer.split('<table>')[1].split('</table>')[0].strip()
		response = answer.split('<table>')[0].strip()
		os.makedirs(f'../model_responses/type1/baselines/{model_name}_chart_to_table/' + image_name + '/' + str(chart_num), exist_ok=True)
		with open(f'../model_responses/type1/baselines/{model_name}_chart_to_table/' + image_name + '/' + str(chart_num) + '/response.txt', 'w', encoding="utf-8") as f:
			f.write(response)
		with open(f'../model_responses/type1/baselines/{model_name}_chart_to_table/' + image_name + '/' + str(chart_num) + '/table.csv', 'w', encoding="utf-8") as f:
			f.write(table)

		del table
		del response
	del outputs
	del index_to_img[image_name]

1188


100%|██████████| 355/355 [46:25<00:00,  7.85s/it] 
